# UNITREE G1 · GR00T-SONIC live 演示

**拿训好的 GR00T-N1.7-SONIC checkpoint，实时 prompt 驱动 G1 跳舞/踢腿/转身走** —— SONIC WBC 当平衡底座，VLA 只出 64 维 latent token。

> 🔧 **前置**：先在 [`BonesSeed_train.ipynb`](BonesSeed_train.ipynb) 跑完 ①GEAR-SONIC WBC 安装 + ②GR00T-N1.7-3B base 下载 + ④record→finetune→eval（出 checkpoint）。本文件只放训好后的 **live 演示**。


## ① 一键 live 演示流（时序 prompt 串接 · 自动起 server · 免 GUI/server 重启）

One running session, **the prompt switches over time** so the G1 does one motion after another
(squat → walk → macarena …) and loops — driven by a **live GR00T server** (闭环,非离线回放).

`gear_sonic_live_demo.sh` 自动**起 GR00T server**(若没起,加载约 1 分钟)再开 viewer,所以是真一键。
server 起一次可复用多个 flow;跑完用 **1.4 停止**释放显存。

> 数据结构 = `scripts/sonic_demo_flows.json`(演示流注册表)。`@flowN` 播注册的流,也可传 `squat,walk,kick` 这种即兴序列。

## ⬇️ 没自己训练?一键下载已发布模型 / one-click download of the published checkpoint

下面的 **GR00T-N1.7** 推理 cell(**1.1 / 1.2 / 1.2a**)会**自动判断**:本地
`outputs/gr00t_sonic_aug2/checkpoint-8000` 存在就用你自己训练的,否则回退到已发布的
[`wsagi/GR00T-N1.7-G1-SONIC-BonesSeed-V2`](https://huggingface.co/wsagi/GR00T-N1.7-G1-SONIC-BonesSeed-V2)
(默认 HF cache `~/.cache/huggingface/hub`)。先跑下面这个下载 cell(幂等,已在 cache 则秒过);
1.2a 的启动 cell 缺失时也会自动下载。

> **V2 = 冷启动修复版**(物理增广 + onset 加权 + stand,kick/walk/jump 可从冷站立自启,零 bootstrap)。
> 训练数据集可从 [`wsagi/SONIC-VLA-BonesSeed-V2`](https://huggingface.co/datasets/wsagi/SONIC-VLA-BonesSeed-V2) 下载。
> 注:**1.2b**(StarVLA)/ **1.2c**(FlowDP)是另两个底座的模型,各自 ckpt 解析见对应脚本,不走这里。

In [ ]:
# 下载 V2 模型到默认 HF cache(已在 cache 则秒过,幂等)。下面 1.1/1.2/1.2a 会自动解析到它。
!dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-BonesSeed-V2'))"

### 1.0 看有哪些演示流

In [ ]:
!bash scripts/gear_sonic_live_demo.sh list

### 1.1 flow1 — 离散动作循环（squat → lunge → macarena）
三个都是**自持动作**:纯 live GR00T 驱动,无 bootstrap,接缝最干净。

In [ ]:
# flow1 三动作循环 — 自动解析 ckpt:优先 outputs/ 自训 aug2-8000,否则用下载的 V2(HF cache)
!CKPT=$PWD/outputs/gr00t_sonic_aug2/checkpoint-8000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-BonesSeed-V2', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的 ⬇️ 下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT bash scripts/gear_sonic_live_demo.sh @flow1

### 1.2 flow2 — deploy 动作循环（squat → jump → dance → walk → lunge → kick → macarena）
串 deploy 动作,**以 squat 开场**,中段 dance→lunge 之间夹一段 walk 过场,循环(macarena 接回 squat),约 **80s/循环**(顺序 2026-06-26 改)。
- **零回放(2026-06-24)**:`flow2` 每段 one-shot 都 `bootstrap_steps:0` = 全程 live GR00T 驱动,不再注入任何 dump token。
  自持动作(squat/lunge/dance/macarena)本来就纯 live;一次性动作(walk/kick/jump)的入场 bootstrap 也去掉了,VLA 必须从静止自启(可能起不来/漂移 = 模型的诚实能力)。
- 唯一非-VLA 窗口 = 接缝 40 步 settle(回中性锚点的 freeze,默认不注入 token,非动作回放)。要连这个也去掉:`SETTLE=0`。
- **每段覆盖率**(JSON `steps` = 纯动作预算,接缝 settle=40 加在外面,百分比 1:1 兑现):squat / lunge / kick / dance = **100%**,walk = **60%**,jump = **48%**,macarena ≈ **15%**。
- 改编排/时长见 `scripts/sonic_demo_flows.json` 的 `flow2`;逐个动作单独看见下方 **1.2a**。

In [ ]:
# flow2 串接 deploy 动作 — 自动解析 ckpt:优先 outputs/ 自训 aug2-8000,否则用下载的 V2(HF cache)
!CKPT=$PWD/outputs/gr00t_sonic_aug2/checkpoint-8000; [ -d "$CKPT" ] || CKPT=$(dependencies/Isaac-GR00T/.venv/bin/python -c "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-BonesSeed-V2', local_files_only=True))" 2>/dev/null) || { echo '先跑上面的 ⬇️ 下载 cell'; exit 1; }; \
 GR00T_CKPT=$CKPT bash scripts/gear_sonic_live_demo.sh @flow2

### 1.2a 持久 live 界面 + 运行时切动作 —— **GR00T-N1.7 头 · 组合 aug2-8000**（点哪个续做哪个,无需重启)
先跑 **▶️ 启动** 一次(后台、不阻塞、有头 GUI);启动后机器人自然站立(走模型 "stand" 微动,非死冻);就绪后**点任一动作单元即时切换**(界面/server 不重启)。点 🧍 回自然站立。
机制:viewer 每几步轮询 `/tmp/sonic_live_prompt.txt`,`sonic_say.sh <motion>` 写新 prompt → 下一步即切(零回放、不复位,VLA 实时驱动)。

**ckpt = `outputs/gr00t_sonic_aug2/checkpoint-8000`**(物理aug剔kick源 + onset加权 + stand 组合)。
**请重点验证冷启动 onset**(启动后/或先点几个动作让机器人漂移,再点 🧍 stand 回站立,再切目标):
- **🚶 walk → 应可靠发起**(实测 n=3 = 3/3,这是物理增广修好的主战果);
- **🦘 jump → 部分发起**(~1/3,起得来时是真跳);
- **🥋 kick → 仍冻**(0/3,最 abrupt 高幅单腿 onset,纯数据没修;kick 重要时把启动单元 `CKPT` 改回 `gr00t_sonic_onsetw/checkpoint-6000`,那个 kick 2/3)。
- 自维持动作(dance/macarena/squat/lunge)切过去能续做。

**结论(诚实)**:物理增广(纯数据)修好 walk + 部分 jump,但 **kick 是盲区**——onset-loss-weighting(onsetw)反而强在 kick。两法互补,**下一步=aug 数据 + ONSET_LOSS_WEIGHT 组合训练**冲三动作全修。完整三模型对比 + 路线见 `.memory/sonic-closeloop-freeze-rootcause.md`。看完用本节末 **⏹️ 关闭** 释放显存。

In [ ]:
# ▶️ 启动持久 live 界面(后台/不阻塞,有头 GUI)。只跑这一次;之后点下面动作即时切换。
# 首次 ~1-2min 载 VLM+Isaac;就绪标志=日志出现 'query #'。查看:  !tail -8 /tmp/sonic_live.log
# ckpt = 组合训练 best aug2-8000(物理增广 + onset 加权 + stand);kick/walk/jump 可从冷站立自启(零 bootstrap,仍有随机性)。
# 启动用哨兵站立(不设 VLA_STAND_MODEL;aug2 的 model-driven stand 数据太少会乱动)。详见 .memory/sonic-closeloop-freeze-rootcause.md。
import os, subprocess
PROMPT_FILE = "/tmp/sonic_live_prompt.txt"
open(PROMPT_FILE, "w").write("stand")                       # 控制文件初值=站立(哨兵 token,不查模型)
# ckpt 自动解析:优先本地自训 aug2-8000,否则下载已发布 V2(首次 ~6GB,之后秒过)。
CKPT = os.path.abspath("outputs/gr00t_sonic_aug2/checkpoint-8000")
if not os.path.isdir(CKPT):
    print("自训 ckpt 缺失 → 下载已发布 V2 …")
    CKPT = subprocess.run(
        ["dependencies/Isaac-GR00T/.venv/bin/python", "-c",
         "from huggingface_hub import snapshot_download; print(snapshot_download('wsagi/GR00T-N1.7-G1-SONIC-BonesSeed-V2'))"],
        capture_output=True, text=True).stdout.strip()
env = {**os.environ, "VLA_PROMPT_FILE": PROMPT_FILE, "GR00T_CKPT": CKPT,
       "LOOPS": "600", "BOOTSTRAP_STEPS": "0"}
subprocess.Popen(["bash", "scripts/gear_sonic_live_demo.sh", "squat"],
                 env=env, stdout=open("/tmp/sonic_live.log", "w"),
                 stderr=subprocess.STDOUT, start_new_session=True)   # 后台,不阻塞(jupyter 不支持 !...&)
print(f"启动中(ckpt={CKPT})… 等 'query #' 再点动作:  !tail -8 /tmp/sonic_live.log")

In [ ]:
# 🏋️ squat — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh squat

In [ ]:
# 🦘 jump — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh jump

In [ ]:
# 🕺 dance — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh dance

In [ ]:
# 🦵 lunge — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh lunge

In [ ]:
# 🚶 walk — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh walk

In [ ]:
# 🥋 kick — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh kick

In [ ]:
# 🕺 macarena — 切到此动作(界面不重启,零回放实时驱动)
!bash scripts/sonic_say.sh macarena

In [ ]:
# 🧍 站立/停下当前动作(回到 idle,机器人站着不动;界面不重启)
!bash scripts/sonic_say.sh stand

In [ ]:
# ⏹️ 关闭持久 live 界面(杀 viewer+server,释放显存)
!bash scripts/gear_sonic_stop.sh

### 1.2b flow2 — **StarVLA GR00T_v2 best**（select_layer sweep 胜出的 sl14）
同一条 `@flow2`，token 来自本次训的 `StarVLA-Qwen3.5-4B-GR00T_v2` **best = sl14**（GR00T N1.7 头设计 +
中层特征 **read@14** + GR00T-faithful 截断 + 解冻 **10-13** 层），经 `serve_starvla_sonic.py`（:5556）驱动。
**为何 sl14**：固定头/截断/解冻4层，只移 select_layer 做了 ±平移扫描（read@10/12/14/15），**read@14 最优**
（读取层 10→12→14 单调改善、14 见顶，再深+加宽到6层反退）。

- **开环 token-MSE = macro 0.0236 / frame 0.0214**（best `sl14/steps_6000` / 6.3ep；skill **1.6×**、**7/7 全破** per-motion-mean 模板，4 个变体里唯一全破）。
  比基线 read@12（0.0243）好 2.6%；但仍比 GR00T N1.7 的 0.0011 差 **~19×**、FlowDP 0.00059 差 36×
  —— = 通用冻骨干（Qwen vs GR00T 的 Cosmos）+ head **从零**（无机器人预训练）的架构天花板，**非** select_layer 能撼动（全 4 方仅差 ~4.5%）。
- 单帧 no-memory：长动作（macarena）幅度可能趋小/漂移。无 FlowDP 那个 gravity-bug（serve `_norm` 对退化维 mask→0）。
- 一键：默认 best `sl14/steps_6000`（其他变体先 `merge_ckpt base delta out` 重建，再 `STARVLA_CKPT=...` 指过去）。跑完用 **1.4** 停。

In [ ]:
# StarVLA GR00T_v2 best (sl14: read@14, unfreeze 10-13) 驱动 flow2 — 脚本默认即 best
!bash scripts/starvla_sonic_gr00t_v2_demo.sh @flow2

### 1.2c 每个动作单独 live —— **FlowDP 头**实时推理(逐个点开看)
每格起/复用 **FlowDP** server(:5557,首次约 20s)+ Isaac viewer,跑**单个动作的实时闭环推理**(非回放)。
FlowDP = Diffusion-Policy 骨干把扩散头换成 rectified-flow(无 VLM,263M);动作选择无语言通路 →
**7 维 motion-onehot 拼进 `observation.state`** 区分 7 动作(详见 `dependencies/FlowHeads/doc/flowdp_diffusion_to_flow_matching.html`)。
- **开环 token-MSE = 0.00059**(best `outputs/flowdp_sonic/checkpoints/last` / ~100ep),比 per-motion-mean 模板好 **65×**、
  7/7 全破,**比 GR00T 的 0.0011 还低 ~2×、比 1.2b 的 StarVLA GR00T_v2 0.0215 低 ~36×**(同 raw token 口径)。排行榜 `outputs/flowdp_sonic/leaderboard.md`。
逐个点开:关 viewer 窗口或中断本格即停该动作;server 复用到下一格;全部看完用 **1.4** 停服务器释放显存。
> ✅ gravity-bug 已修 → **不再倒地**(根因:数据集 projected_gravity 全零→退化归一化把 live gravity 放大成 ~1e8→爆;serve 已置零修复)。
> ⚠️ 单帧 no-memory:初始动一下后幅度可能趋小(模型缺相位/速度信号)。治本 = 历史条件重训(n_obs_steps≥2)。

In [ ]:
# 🏋️ squat 深蹲 — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh squat

In [ ]:
# 🦘 jump 单腿跳 — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh jump

In [ ]:
# 🕺 dance — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh dance

In [ ]:
# 🦵 lunge 弓步 — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh lunge

In [ ]:
# 🥋 kick 踢腿 — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh kick

In [ ]:
# 🚶 walk 转身走 — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh walk

In [ ]:
# 🕺 macarena — flowdp 实时推理
!bash scripts/gear_sonic_flowdp_demo.sh macarena

### 1.3 即兴序列 + 视角/停止

**即兴序列**(逗号分隔,可 `名:步数`):

**viewer 快捷键**(先点一下视口获得焦点):`C` = 一键贴近机器人 · `F` = 跟随/自由切换 · `R` = 复位。

**视角微调**(relaunch 时加前缀):`CAM_EYE="x,y,z"` 相机位 · `CAM_LOOKAT="x,y,z"` 瞄准点
(lookat 第 3 位往下=多看腿脚;eye 前两位缩小=拉近)。每段时长 `STEPS=240`,接缝过渡 `SETTLE=40`。

In [ ]:
!bash scripts/gear_sonic_live_demo.sh squat,lunge,macarena

### 1.4 停止 live 演示（杀 viewer + GR00T server,释放显存）

In [ ]:
!bash scripts/gear_sonic_stop.sh